# Self-RAG from Scratch — LangChain + Gemini

This notebook implements **Self-RAG** (*Self-Reflective Retrieval-Augmented Generation*, Asai et al., 2023) using LangChain and Google's Gemini chat models.

### The core idea of the paper
Standard RAG always retrieves a fixed number of passages and stuffs them into the prompt, regardless of whether retrieval is even needed, and without checking whether the retrieved text actually supports the final answer. Self-RAG fixes this by training a single LM to interleave generation with **self-reflection**, expressed as special "reflection tokens":

| Token | Question it answers | Values |
|---|---|---|
| `Retrieve` | Do I need to retrieve for this query/segment? | `Yes` / `No` / `Continue` |
| `ISREL` | Is this retrieved passage relevant to the query? | `Relevant` / `Irrelevant` |
| `ISSUP` | Is the generated text actually supported by the passage? | `Fully Supported` / `Partially Supported` / `No Support` |
| `ISUSE` | How useful is this candidate response overall? | `1`–`5` |

### The paper's inference-time flow
```
query
  └─ Retrieve? ──No──► generate answer directly (no retrieval)
       │Yes
       ▼
  retrieve top-k passages
       │
       ├─ for each passage:
       │     ISREL(passage, query)  ──Irrelevant──► discard
       │     │Relevant
       │     generate candidate segment conditioned on passage
       │     ISSUP(passage, segment)
       │     ISUSE(query, segment)
       │     score = f(ISREL, ISSUP, ISUSE)          <- "segment-level beam search"
       │
       └─ pick highest-scoring segment as the output
```
The original paper does this token-by-token / segment-by-segment with a single model that was **fine-tuned** to emit these tokens as part of its vocabulary, and it uses the tokens' output probabilities to compute a soft weighted score.


### What this notebook does differently (and why that's fine for learning)

- **No fine-tuned critic model.** We don't have a Self-RAG-tuned checkpoint, so we *simulate* each reflection token with a separate prompted call to Gemini (`gemini` acting as `Retrieve` / `ISREL` / `ISSUP` / `ISUSE` classifier). Architecturally this preserves the paper's control flow; it just replaces "one model, special tokens, token logits" with "one model, structured prompts, discrete text labels".
- **Discrete scores instead of token probabilities.** The paper's segment score is a weighted sum over the *softmax probability* of the critique tokens. We don't have access to Gemini's token-level logits for arbitrary labels, so we map each label to a fixed numeric weight instead. The ranking logic is otherwise the same.
- **Small, hand-built corpus.** The paper trains/evaluates on large-scale QA datasets. We use a handful of short documents so every retrieval and critique step stays easy to inspect.
- **Single-pass segments.** The paper can do this recursively over multiple generation segments per passage (a mini beam search). We do one candidate generation per retrieved passage, then pick the best passage's answer — this keeps the notebook readable while keeping the *select-the-best-supported-candidate* idea intact.

Everything else — the adaptive retrieval gate, per-passage relevance filtering, support checking, and utility-based selection — mirrors the paper's pipeline.


## 1. Install dependencies


In [1]:
!pip install -qU langchain langchain-google-genai langchain-community langchain-chroma chromadb


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


## 2. Configure the Gemini API key

You need a Google AI Studio API key (https://aistudio.google.com/apikey). It's read securely at runtime rather than hard-coded.


In [2]:
import os
import getpass

if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google AI Studio API key: ")

## 3. Build a niche knowledge base

We use **fictional / highly-specific documents** that a general-purpose LLM
would NOT know from training — forcing the `Retrieve?` gate to answer **Yes**.
The corpus is intentionally varied so we can observe every Self-RAG path:

| Document | Topic |
|---|---|
| `orion_ai_product` | Orion AI's Aether-X federated learning platform (pricing, limits) |
| `aether_troubleshooting` | Aether-X error codes and CLI fixes |
| `nexavin_clinical` | Phase II results for fictional drug Nexavin |
| `bluehaven_zoning` | Bluehaven city drone permit code §4.7.2 |
| `solarcup_2024` | International Solar Cup 2024 race results |
| `zeta_benchmark` | Zeta Benchmark 2024 LLM leaderboard stats |

Having **unrelated topics** lets `ISREL` reject off-topic passages; having
**two docs on the same topic** (Aether-X) demonstrates ranked candidate selection.


In [4]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma

# ── Niche, fictional documents (all content is invented) ─────────────────
raw_docs = [

    # 1. Orion AI — product specs (pricing, limits, OS requirements)
    Document(
        page_content=(
            "Orion AI was founded in 2023 by Dr. Priya Nair. Its flagship product "
            "Aether-X is a federated learning framework supporting up to 500 nodes "
            "with a maximum throughput of 12 TB/hour. The Enterprise tier costs "
            "$4,200/month and includes 24/7 SLA support. The Free tier is capped at "
            "5 nodes and 100 GB/month. Aether-X runs on Linux (kernel >= 5.15) and "
            "requires Python 3.10+."
        ),
        metadata={"source": "orion_ai_product"},
    ),

    # 2. Aether-X — troubleshooting guide (error codes, CLI fix, log path)
    Document(
        page_content=(
            "Aether-X known issues: Node connection failures (error AX-404) are fixed "
            "by setting heartbeat_timeout=30 in aether.config. Memory overflow errors "
            "(AX-512) occur when gradient_accumulation_steps > 64 on nodes with less "
            "than 16 GB RAM — reduce the value to 32. Run 'aether-x diagnose --full' "
            "to generate a health report. Logs are stored in /var/log/aether/."
        ),
        metadata={"source": "aether_troubleshooting"},
    ),

    # 3. Nexavin — Phase II clinical trial data (ORR, side effects, dosage)
    Document(
        page_content=(
            "Nexavin (nexavimab-blt) is an investigational monoclonal antibody by "
            "HorizonPharma. In the Phase II NEXUS-2 trial (n=312), Nexavin achieved a "
            "67.4% objective response rate vs 21.3% for placebo (p<0.001). Common "
            "adverse events: fatigue (38%), nausea (24%), peripheral neuropathy (11%). "
            "Recommended dose: 8 mg/kg IV every 3 weeks. "
            "Phase III trials have not yet been initiated as of Q2 2025."
        ),
        metadata={"source": "nexavin_clinical"},
    ),

    # 4. Bluehaven — municipal drone permit code
    Document(
        page_content=(
            "Bluehaven Municipal Code section 4.7.2 (amended July 2024): commercial "
            "drone operations require a Class-C airspace waiver AND a local operator "
            "permit (LOP-17 form). LOPs cost $350/year and must be renewed by March 31. "
            "Residential properties within 400 feet of a drone landing zone are entitled "
            "to a free noise impact assessment. Violations carry a fine of $1,500/day."
        ),
        metadata={"source": "bluehaven_zoning"},
    ),

    # 5. Solar Cup 2024 — race results, speeds, vehicle specs
    Document(
        page_content=(
            "At the International Solar Cup 2024 (Seville, Spain), Team Helios (Norway) "
            "won with an average speed of 94.3 km/h over the 1,200 km course. "
            "Team Aurora (Canada) finished second at 91.7 km/h. The race ran 8 stages "
            "over 5 days. Team Helios used a 1.2 kWh lithium-ceramic battery and "
            "4.2 m2 of bifacial solar panels producing 820 W peak."
        ),
        metadata={"source": "solarcup_2024"},
    ),

    # 6. Zeta Benchmark 2024 — LLM leaderboard with exact scores
    Document(
        page_content=(
            "The Zeta Benchmark 2024 report (Chen et al.) evaluated 14 open-source LLMs. "
            "Llama-3-70B scored 81.2% on MMLU, 74.6% on HumanEval, 68.9% on ARC-Challenge. "
            "Phi-3-mini (3.8B parameters) achieved 72.1% MMLU, the best efficiency ratio. "
            "Mistral-22B ranked third overall. "
            "The benchmark will not be updated until 2026."
        ),
        metadata={"source": "zeta_benchmark"},
    ),
]

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(raw_docs)

embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")
vectorstore = Chroma.from_documents(chunks, embedding=embeddings, collection_name="self_rag_demo")
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

print(f"Indexed {len(chunks)} chunks from {len(raw_docs)} documents.")

Indexed 6 chunks from 6 documents.


In [5]:
chunks

[Document(metadata={'source': 'orion_ai_product'}, page_content='Orion AI was founded in 2023 by Dr. Priya Nair. Its flagship product Aether-X is a federated learning framework supporting up to 500 nodes with a maximum throughput of 12 TB/hour. The Enterprise tier costs $4,200/month and includes 24/7 SLA support. The Free tier is capped at 5 nodes and 100 GB/month. Aether-X runs on Linux (kernel >= 5.15) and requires Python 3.10+.'),
 Document(metadata={'source': 'aether_troubleshooting'}, page_content="Aether-X known issues: Node connection failures (error AX-404) are fixed by setting heartbeat_timeout=30 in aether.config. Memory overflow errors (AX-512) occur when gradient_accumulation_steps > 64 on nodes with less than 16 GB RAM — reduce the value to 32. Run 'aether-x diagnose --full' to generate a health report. Logs are stored in /var/log/aether/."),
 Document(metadata={'source': 'nexavin_clinical'}, page_content='Nexavin (nexavimab-blt) is an investigational monoclonal antibody b

## 4. Set up the Gemini chat model

One `ChatGoogleGenerativeAI` instance plays every role in the pipeline (retrieval-decider, relevance critic, generator, support critic, utility critic) — exactly as in the paper a *single* LM emits all reflection tokens plus the generation itself. We just call it with different prompts for different roles instead of different special tokens.


In [6]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

## 5. Reflection token 1 — `Retrieve`: do we even need to retrieve?

Self-RAG's biggest efficiency win over vanilla RAG is that retrieval is **conditional**. Simple factual/commonsense queries ("what's 12 x 12?") don't need a search; knowledge-heavy queries do. This function stands in for that gate.


In [7]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

retrieve_decision_prompt = ChatPromptTemplate.from_template(
    "You are the retrieval-decision module of a Self-RAG pipeline.\n"
    "Decide whether answering the query well requires retrieving external documents, "
    "or whether it can be answered reliably from general knowledge/reasoning alone.\n\n"
    "Query: {query}\n\n"
    "Respond with exactly one word: Yes (retrieval needed) or No (retrieval not needed)."
)

def decide_retrieve(query: str) -> str:
    chain = retrieve_decision_prompt | llm | StrOutputParser()
    return chain.invoke({"query": query}).strip().split()[0]

## 6. Reflection token 2 — `ISREL`: is this passage relevant?

After retrieving candidate passages, the paper checks *each one individually* rather than trusting the retriever's ranking blindly. Irrelevant passages are dropped before any generation happens on top of them.


In [8]:
isrel_prompt = ChatPromptTemplate.from_template(
    "Passage:\n{passage}\n\n"
    "Query: {query}\n\n"
    "Does this passage contain information relevant to answering the query?\n"
    "Respond with exactly one word: Relevant or Irrelevant."
)

def critique_isrel(query: str, passage: str) -> str:
    chain = isrel_prompt | llm | StrOutputParser()
    return chain.invoke({"query": query, "passage": passage}).strip().split()[0]

## 7. Generate a candidate answer conditioned on the passage

For every passage that survives the `ISREL` filter, Self-RAG generates a candidate response segment grounded in that specific passage. Note we generate **one candidate per surviving passage**, not one candidate for all passages combined — this per-passage grounding is what makes the later support check ( `ISSUP` ) meaningful.


In [9]:
generate_prompt = ChatPromptTemplate.from_template(
    "Answer the query using only the information in the passage below. "
    "Be concise and do not add outside facts.\n\n"
    "Passage:\n{passage}\n\n"
    "Query: {query}\n\n"
    "Answer:"
)

def generate_candidate(query: str, passage: str) -> str:
    chain = generate_prompt | llm | StrOutputParser()
    return chain.invoke({"query": query, "passage": passage}).strip()

## 8. Reflection token 3 — `ISSUP`: is the answer actually supported?

This is the hallucination check: given the passage and the generated answer, how much of the answer's content is actually backed by the passage versus made up?


In [10]:
issup_prompt = ChatPromptTemplate.from_template(
    "Passage:\n{passage}\n\n"
    "Generated answer:\n{answer}\n\n"
    "Does the passage provide enough information to answer the query, "
    "and does the generated answer reflect that?\n"
    "IMPORTANT: If the generated answer says the passage does not contain information "
    "or cannot answer the query, you MUST respond with: No Support\n"
    "Respond with exactly one label: Fully Supported, Partially Supported, or No Support."
)

def critique_issup(passage: str, answer: str) -> str:
    chain = issup_prompt | llm | StrOutputParser()
    return chain.invoke({"passage": passage, "answer": answer}).strip()

## 9. Reflection token 4 — `ISUSE`: how useful is this candidate overall?

Support alone isn't enough — a fully-supported but unhelpful one-word answer shouldn't win. `ISUSE` scores overall usefulness to the query on a 1-5 scale, same as the paper.


In [11]:
isuse_prompt = ChatPromptTemplate.from_template(
    "Query: {query}\n\n"
    "Candidate answer: {answer}\n\n"
    "Rate how useful this answer is for the query, from 1 (not useful) to 5 (extremely useful).\n"
    "Respond with only the integer."
)

def critique_isuse(query: str, answer: str) -> str:
    chain = isuse_prompt | llm | StrOutputParser()
    raw = chain.invoke({"query": query, "answer": answer}).strip()
    digits = "".join(ch for ch in raw if ch.isdigit())
    return digits[:1] if digits else "1"

## 10. Score and rank candidates (the paper's "segment-level beam search")

The paper ranks candidate segments by a weighted combination of the critique tokens' probabilities. We approximate this by mapping each discrete label to a fixed weight and summing them — same ranking behavior, simpler math since we don't have access to Gemini's token probabilities for these custom labels.


In [12]:
REL_WEIGHTS = {"relevant": 1.0, "irrelevant": 0.0}
SUP_WEIGHTS = {"fully supported": 1.0, "partially supported": 0.5, "no support": 0.0}

def compute_segment_score(isrel: str, issup: str, isuse: str) -> float:
    rel_score = REL_WEIGHTS.get(isrel.lower(), 0.0)
    sup_score = SUP_WEIGHTS.get(issup.lower(), 0.0)
    try:
        use_score = int(isuse) / 5.0
    except ValueError:
        use_score = 0.0
    # Equal-weighted sum, matching the paper's additive critique score.
    return rel_score + sup_score + use_score

## 11. Put it all together: the Self-RAG pipeline

This orchestrates the full flow from the diagram at the top:
1. `Retrieve?` gate — skip straight to generation if retrieval isn't needed.
2. Retrieve top-k passages.
3. For each passage: `ISREL` filter → generate candidate → `ISSUP` + `ISUSE` → score.
4. Return the highest-scoring candidate (or a fallback if nothing was relevant).

A `trace` list is returned alongside the answer purely so we can inspect every reflection-token decision the pipeline made — the paper's whole point is that this reasoning is auditable, not a black box.


In [13]:
REFUSAL_PHRASES = [
    "does not contain", "does not have", "no information",
    "cannot answer", "not mentioned", "not provided",
    "i cannot", "the passage does not", "passage does not",
    "not enough information", "insufficient information",
]

def is_refusal(answer: str) -> bool:
    """Return True if the LLM signalled it could not answer from the passage."""
    lower = answer.lower()
    return any(phrase in lower for phrase in REFUSAL_PHRASES)

def self_rag(query: str, k: int = 4):
    trace = {"query": query, "retrieve_decision": None, "candidates": [], "mode": None}

    retrieve_decision = decide_retrieve(query)
    trace["retrieve_decision"] = retrieve_decision

    if retrieve_decision.lower() == "no":
        trace["mode"] = "direct_generation"
        answer = llm.invoke(query).content
        return answer, trace

    trace["mode"] = "retrieval_augmented"
    docs = retriever.invoke(query)[:k]

    scored_candidates = []
    for doc in docs:
        passage = doc.page_content
        isrel = critique_isrel(query, passage)

        if isrel.lower() != "relevant":
            trace["candidates"].append({
                "source": doc.metadata.get("source"),
                "isrel": isrel,
                "kept": False,
                "passage": passage,          # ← show chunk in trace
            })
            continue

        answer = generate_candidate(query, passage)

        # Fix 1: detect refusal answers before running ISSUP/ISUSE
        if is_refusal(answer):
            trace["candidates"].append({
                "source": doc.metadata.get("source"),
                "isrel": isrel,
                "issup": "No Support",
                "isuse": "1",
                "score": 0.0,
                "answer": answer,
                "kept": False,               # discard — model said "no info"
                "passage": passage,          # ← show chunk in trace
            })
            continue

        issup = critique_issup(passage, answer)
        isuse = critique_isuse(query, answer)
        score = compute_segment_score(isrel, issup, isuse)

        trace["candidates"].append({
            "source": doc.metadata.get("source"),
            "isrel": isrel,
            "issup": issup,
            "isuse": isuse,
            "score": round(score, 2),
            "answer": answer,
            "kept": True,
            "passage": passage,              # ← show chunk in trace
        })
        scored_candidates.append((score, answer))

    if not scored_candidates:
        trace["mode"] = "retrieval_augmented_fallback"
        answer = llm.invoke(query).content
        return answer, trace

    scored_candidates.sort(key=lambda x: x[0], reverse=True)
    best_score, best_answer = scored_candidates[0]
    trace["selected_score"] = round(best_score, 2)
    return best_answer, trace

## 12. Try it out — 5 queries covering every Self-RAG path


In [14]:
import json

def show_result(query: str):
    answer, trace = self_rag(query)
    print("\nQUERY:", query)
    print("-" * 70)
    print(json.dumps(trace, indent=2, default=str))
    print("-" * 70)
    print("FINAL ANSWER:", answer)
    print("=" * 70)

# ── Q1: Retrieve=Yes, doc is Relevant + Fully Supported ───────────────────
# Expected: nexavin_clinical is retrieved, passes ISREL, generates a clean
# answer, ISSUP=Fully Supported, high ISUSE, selected as best candidate.
show_result("What was the objective response rate for Nexavin in its Phase II trial, and what are its common side effects?")



QUERY: What was the objective response rate for Nexavin in its Phase II trial, and what are its common side effects?
----------------------------------------------------------------------
{
  "query": "What was the objective response rate for Nexavin in its Phase II trial, and what are its common side effects?",
  "retrieve_decision": "Yes",
  "candidates": [
    {
      "source": "nexavin_clinical",
      "isrel": "Relevant",
      "issup": "Fully Supported",
      "isuse": "5",
      "score": 3.0,
      "answer": "The objective response rate for Nexavin in its Phase II trial was 67.4%. Its common side effects are fatigue, nausea, and peripheral neuropathy.",
      "kept": true,
      "passage": "Nexavin (nexavimab-blt) is an investigational monoclonal antibody by HorizonPharma. In the Phase II NEXUS-2 trial (n=312), Nexavin achieved a 67.4% objective response rate vs 21.3% for placebo (p<0.001). Common adverse events: fatigue (38%), nausea (24%), peripheral neuropathy (11%). Recomme

In [15]:
# ── Q2: Retrieve=Yes + ISREL filtering ────────────────────────────────────
# Expected: solarcup_2024 is Relevant; the other 3 retrieved docs
# (e.g. nexavin, orion_ai, zeta_benchmark) are Irrelevant and filtered out.
show_result("Who won the International Solar Cup 2024 and what was their average speed?")



QUERY: Who won the International Solar Cup 2024 and what was their average speed?
----------------------------------------------------------------------
{
  "query": "Who won the International Solar Cup 2024 and what was their average speed?",
  "retrieve_decision": "Yes",
  "candidates": [
    {
      "source": "solarcup_2024",
      "isrel": "Relevant",
      "issup": "Fully Supported",
      "isuse": "5",
      "score": 3.0,
      "answer": "Team Helios (Norway) won with an average speed of 94.3 km/h.",
      "kept": true,
      "passage": "At the International Solar Cup 2024 (Seville, Spain), Team Helios (Norway) won with an average speed of 94.3 km/h over the 1,200 km course. Team Aurora (Canada) finished second at 91.7 km/h. The race ran 8 stages over 5 days. Team Helios used a 1.2 kWh lithium-ceramic battery and 4.2 m2 of bifacial solar panels producing 820 W peak."
    },
    {
      "source": "zeta_benchmark",
      "isrel": "Irrelevant",
      "kept": false,
      "passage":

In [16]:
# ── Q4: Retrieve=No — direct generation ──────────────────────────────────
# Expected: decide_retrieve() returns 'No'; LLM answers directly without
# hitting the vector store at all. mode='direct_generation' in trace.
show_result("What is 47 multiplied by 13?")


QUERY: What is 47 multiplied by 13?
----------------------------------------------------------------------
{
  "query": "What is 47 multiplied by 13?",
  "retrieve_decision": "No",
  "candidates": [],
  "mode": "direct_generation"
}
----------------------------------------------------------------------
FINAL ANSWER: To multiply 47 by 13, we can break it down:

1.  Multiply 47 by 10:
    47 * 10 = 470

2.  Multiply 47 by 3:
    47 * 3 = (40 * 3) + (7 * 3) = 120 + 21 = 141

3.  Add the results from step 1 and step 2:
    470 + 141 = 611

So, 47 multiplied by 13 is **611**.


## 13. Recap: mapping back to the paper

| Paper concept | This notebook |
|---|---|
| Single fine-tuned LM emits `Retrieve`/`ISREL`/`ISSUP`/`ISUSE` as special tokens during generation | One Gemini model called with different prompts for each role |
| Token probabilities weight the segment score | Fixed weights per discrete label |
| Adaptive, on-demand retrieval | `decide_retrieve()` gate before hitting the vector store |
| Per-passage relevance filtering | `critique_isrel()` before generation |
| Hallucination/groundedness check | `critique_issup()` |
| Utility-based reranking of candidates | `critique_isuse()` + `compute_segment_score()` |
| Segment-level beam search over multiple continuations | Simplified to one candidate per passage, best-of-k selection |

### Natural extensions if you want to go further
- Generate **multiple** candidate continuations per passage (real beam search) and score each.
- Ask the `ISUSE`/`ISSUP` prompts to also return a short justification, and log it in `trace` for inspection.
- Swap the equal-weighted sum in `compute_segment_score` for a tunable weighted sum, and tune weights against a small labeled eval set.
- Replace the single flat retrieval with iterative retrieval (re-retrieve if `ISUSE` is low), closer to the paper's segment-by-segment loop.
